# DEPRECATED — v1.0 Research Baseline

This notebook is a **frozen historical reference** of the original v1.0 experiment.
It used Hugging Face Hub downloads and BitsAndBytes NF4 quantization.

**Do not run this notebook for new experiments.**
Use `qwen-selective-regeneration-v2.ipynb` instead, which loads
Qwen2.5-Coder from Kaggle with GPTQ INT4 quantization.

## Cell 1 — Hardware and environment check

In [ ]:
import os
import platform
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())

subprocess.run(["nvidia-smi"], check=False)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print("VRAM GB:", round(props.total_memory / 1024**3, 2))
except Exception as exc:
    print("Torch inspection failed:", exc)

## Cell 2 — Install dependencies

In [ ]:
!pip install -q -U \
    "transformers>=4.48,<5" \
    "accelerate>=1.2" \
    "sentencepiece>=0.2" \
    "safetensors>=0.4" \
    "pydantic>=2.8,<=2.12.3" \
    "pyyaml>=6.0" \
    "jsonschema>=4.23" \
    "pandas>=2.2" \
    "ruff>=0.11" \
    "gptqmodel>=2.0" \
    "bitsandbytes>=0.45" \
    "psutil>=5.9"

In [ ]:
!pip install -q fastapi httpx pytest pytest-cov

## Cell 3 — Verify package versions

In [ ]:
import accelerate
import jsonschema
import pandas
import pydantic
import torch
import transformers
import yaml

VERSIONS = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "accelerate": accelerate.__version__,
    "pydantic": pydantic.__version__,
    "jsonschema": jsonschema.__version__,
    "yaml": yaml.__version__,
    "pandas": pandas.__version__,
}

try:
    import gptqmodel
    VERSIONS["gptqmodel"] = gptqmodel.__version__
except ImportError:
    VERSIONS["gptqmodel"] = "not installed"

try:
    import bitsandbytes
    VERSIONS["bitsandbytes"] = bitsandbytes.__version__
except ImportError:
    VERSIONS["bitsandbytes"] = "not installed"

try:
    import psutil
    VERSIONS["psutil"] = psutil.__version__
except ImportError:
    VERSIONS["psutil"] = "not installed"

for name, version in VERSIONS.items():
    print(f"{name:20s} {version}")

## Cell 4 — Create directories and set up module path

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/selective_regeneration")

PATHS = {
    "root": ROOT,
    "benchmark": ROOT / "benchmark",
    "runtime": ROOT / "runtime",
    "snapshots": ROOT / "snapshots",
    "results": ROOT / "results",
    "prompts": ROOT / "prompts",
    "schemas": ROOT / "schemas",
    "logs": ROOT / "logs",
}

for key, path in PATHS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"  {key:12s} -> {path}")

sys.path.insert(0, str(ROOT))
print(f"\nModule path added: {ROOT}")

## Cell 5 — Experiment configuration

In [ ]:
from dataclasses import asdict, dataclass
import json


@dataclass(frozen=True)
class ExperimentConfig:
    experiment_name: str = "todo_selective_regeneration_v1"

    model_source: str = "kaggle"
    model_id: str = "qwen-lm/qwen2.5-coder"
    model_path: str = ""
    quantization_backend: str = "gptq"

    max_context_tokens: int = 8192
    max_new_tokens: int = 4000

    do_sample: bool = False
    seed: int = 42

    max_repair_attempts: int = 1
    generation_timeout_seconds: int = 600
    test_timeout_seconds: int = 180

    run_ruff: bool = True
    run_unit_tests: bool = True
    run_integration_tests: bool = True
    run_hidden_tests: bool = True


CONFIG = ExperimentConfig()

config_path = PATHS["root"] / "experiment_config.json"
config_path.write_text(
    json.dumps(asdict(CONFIG), indent=2),
    encoding="utf-8",
)

print(config_path.read_text())

## Cell 6 — Import reusable modules

In [ ]:
from selective_regeneration.change_model import RequirementChange
from selective_regeneration.dsl import read_yaml, write_yaml, apply_dsl_patch
from selective_regeneration.diff import flatten_dict, structural_diff
from selective_regeneration.dependency import resolve_affected_files
from selective_regeneration.validation import validate_project, run_command, stage_passed
from selective_regeneration.deterministic_repair import run_deterministic_local_repair
from selective_regeneration.patching import write_files, read_selected_files, apply_generated_payload
from selective_regeneration.snapshots import hash_file, build_manifest, create_snapshot, compare_manifests
from selective_regeneration.context_builder import (
    SYSTEM_PROMPT, REPAIR_SYSTEM_PROMPT, GENERATION_SCHEMA,
    build_generation_prompt, build_repair_prompt, summarize_validation_failures,
)
from selective_regeneration.output_parser import (
    strip_markdown_fence, extract_json_object, validate_payload_schema, validate_generated_scope,
)
from selective_regeneration.metrics import build_change_metrics, append_metrics_jsonl, load_metrics_jsonl
from selective_regeneration.llm_runner import (
    GenerationResult, load_model, generate_with_qwen,
    find_kaggle_model,
)
from selective_regeneration.experiment_runner import run_selective_change

print("All modules imported successfully.")

## Cell 7 — Create initial project

In [ ]:
PROJECT_DIR = PATHS["runtime"] / "todo_api"

initial_files = {
    "app/__init__.py": "",
    "app/models.py": """
from dataclasses import dataclass
from uuid import UUID


@dataclass
class Task:
    id: UUID
    title: str
    completed: bool = False
""",
    "app/schemas.py": """
from uuid import UUID

from pydantic import BaseModel, Field


class TaskCreate(BaseModel):
    title: str = Field(min_length=1)


class TaskResponse(BaseModel):
    id: UUID
    title: str
    completed: bool
""",
    "app/repository.py": """
from uuid import UUID

from app.models import Task


class TaskRepository:
    def __init__(self) -> None:
        self._tasks: dict[UUID, Task] = {}

    def save(self, task: Task) -> Task:
        self._tasks[task.id] = task
        return task

    def get(self, task_id: UUID) -> Task:
        if task_id not in self._tasks:
            raise KeyError(f"Task {task_id} not found")
        return self._tasks[task_id]

    def list_all(self) -> list[Task]:
        return list(self._tasks.values())

    def clear(self) -> None:
        self._tasks.clear()
""",
    "app/service.py": """
from uuid import UUID, uuid4

from app.models import Task
from app.repository import TaskRepository


class TaskService:
    def __init__(self, repository: TaskRepository) -> None:
        self.repository = repository

    def create_task(self, title: str) -> Task:
        normalized_title = title.strip()
        if not normalized_title:
            raise ValueError("Title must not be empty")

        task = Task(
            id=uuid4(),
            title=normalized_title,
            completed=False,
        )
        return self.repository.save(task)

    def complete_task(self, task_id: UUID) -> Task:
        task = self.repository.get(task_id)
        task.completed = True
        return self.repository.save(task)

    def list_tasks(self) -> list[Task]:
        return self.repository.list_all()
""",
    "app/main.py": """
from uuid import UUID

from fastapi import FastAPI, HTTPException

from app.repository import TaskRepository
from app.schemas import TaskCreate, TaskResponse
from app.service import TaskService


app = FastAPI(title="ToDo API")

repository = TaskRepository()
service = TaskService(repository)


@app.get("/health")
def health() -> dict[str, str]:
    return {"status": "ok"}


@app.post("/tasks", response_model=TaskResponse, status_code=201)
def create_task(payload: TaskCreate) -> TaskResponse:
    try:
        task = service.create_task(payload.title)
        return TaskResponse.model_validate(task, from_attributes=True)
    except ValueError as exc:
        raise HTTPException(status_code=422, detail=str(exc)) from exc


@app.get("/tasks", response_model=list[TaskResponse])
def list_tasks() -> list[TaskResponse]:
    return [
        TaskResponse.model_validate(task, from_attributes=True)
        for task in service.list_tasks()
    ]


@app.post("/tasks/{task_id}/complete", response_model=TaskResponse)
def complete_task(task_id: UUID) -> TaskResponse:
    try:
        task = service.complete_task(task_id)
        return TaskResponse.model_validate(task, from_attributes=True)
    except KeyError as exc:
        raise HTTPException(status_code=404, detail=str(exc)) from exc
""",
    "tests/conftest.py": """
import pytest

from app.main import repository


@pytest.fixture(autouse=True)
def clear_repository() -> None:
    repository.clear()
""",
    "tests/unit/test_service.py": """
import pytest

from app.repository import TaskRepository
from app.service import TaskService


def make_service() -> TaskService:
    return TaskService(TaskRepository())


def test_create_task_success() -> None:
    task = make_service().create_task("Buy milk")

    assert task.title == "Buy milk"
    assert task.completed is False


def test_create_task_rejects_blank_title() -> None:
    with pytest.raises(ValueError):
        make_service().create_task("   ")


def test_complete_task() -> None:
    service = make_service()
    task = service.create_task("Buy milk")

    completed = service.complete_task(task.id)

    assert completed.completed is True
""",
    "tests/integration/test_api.py": """
from fastapi.testclient import TestClient

from app.main import app


client = TestClient(app)


def test_health() -> None:
    response = client.get("/health")

    assert response.status_code == 200
    assert response.json() == {"status": "ok"}


def test_create_and_list_task() -> None:
    create_response = client.post(
        "/tasks",
        json={"title": "Buy milk"},
    )

    assert create_response.status_code == 201

    task = create_response.json()
    assert task["title"] == "Buy milk"
    assert task["completed"] is False

    list_response = client.get("/tasks")

    assert list_response.status_code == 200
    assert len(list_response.json()) == 1
""",
    "hidden_tests/test_initial_hidden.py": """
from fastapi.testclient import TestClient

from app.main import app


client = TestClient(app)


def test_whitespace_title_is_rejected() -> None:
    response = client.post("/tasks", json={"title": "   "})
    assert response.status_code == 422
""",
    "dsl.yaml": """
project:
  id: todo-api
  version: "1.0"

entities:
  Task:
    fields:
      id:
        type: uuid
        required: true
      title:
        type: string
        required: true
        constraints:
          - non_blank
      completed:
        type: boolean
        required: true
        default: false

actions:
  create_task:
    inputs:
      - title
    output: Task

  complete_task:
    inputs:
      - task_id
    output: Task

  list_tasks:
    inputs: []
    output: List[Task]

business_rules:
  title_non_blank:
    text: Task title must contain at least one non-whitespace character.

architecture:
  layers:
    - api
    - service
    - repository
    - domain
""",
    "requirements.txt": """
fastapi
httpx
pydantic
pytest
pytest-cov
ruff
""",
}

if PROJECT_DIR.exists():
    import shutil
    shutil.rmtree(PROJECT_DIR)

write_files(PROJECT_DIR, initial_files)

print(PROJECT_DIR)

## Cell 8 — Validate initial project

In [ ]:
initial_validation = validate_project(
    PROJECT_DIR,
    test_timeout=CONFIG.test_timeout_seconds,
)

print("All passed:", initial_validation["all_passed"])
assert initial_validation["all_passed"], "Initial project validation failed"

## Cell 9 — Save initial snapshot

In [ ]:
initial_snapshot = create_snapshot(
    PROJECT_DIR,
    "v0_initial",
    snapshots_dir=PATHS["snapshots"],
)

print("Initial snapshot:", initial_snapshot)

## Cell 10 — Load tokenizer and model

In [ ]:
discovered = find_kaggle_model()
if discovered:
    print(f"Discovered model at: {discovered}")
else:
    print("No model found in /kaggle/input. Add the model via Kaggle Add Data.")

tokenizer, model = load_model(
    model_path=CONFIG.model_path or str(discovered) if discovered else None,
    model_source=CONFIG.model_source,
    quantization_backend=CONFIG.quantization_backend,
    seed=CONFIG.seed,
)

## Cell 11 — Smoke test

In [ ]:
smoke = generate_with_qwen(
    tokenizer=tokenizer,
    model=model,
    system_prompt="You are a code transformation engine. Return only valid JSON.",
    user_prompt='Return exactly this JSON object: {"status": "ok"}',
    max_new_tokens=100,
    max_context_tokens=CONFIG.max_context_tokens,
)

print("Smoke test tokens:", smoke.total_tokens)
print("Smoke test output:", smoke.text[:200])
assert "ok" in smoke.text.lower(), "Smoke test failed"

## Cell 12 — Dependency rules

In [ ]:
DEPENDENCY_RULES = {
    "entities.Task.fields": {
        "artifacts": [
            "app/models.py",
            "app/schemas.py",
            "app/service.py",
            "tests/unit/test_service.py",
            "tests/integration/test_api.py",
        ]
    },
    "actions.create_task": {
        "artifacts": [
            "app/schemas.py",
            "app/service.py",
            "app/main.py",
            "tests/unit/test_service.py",
            "tests/integration/test_api.py",
        ]
    },
    "actions.list_overdue_tasks": {
        "artifacts": [
            "app/service.py",
            "app/main.py",
            "tests/unit/test_service.py",
            "tests/integration/test_api.py",
        ]
    },
    "business_rules.overdue_rule": {
        "artifacts": [
            "app/service.py",
            "tests/unit/test_service.py",
            "tests/integration/test_api.py",
        ]
    },
    "actions.list_tasks": {
        "artifacts": [
            "app/service.py",
            "app/main.py",
            "tests/unit/test_service.py",
            "tests/integration/test_api.py",
        ]
    },
    "business_rules.tag_filter_rule": {
        "artifacts": [
            "app/service.py",
            "app/main.py",
            "tests/unit/test_service.py",
            "tests/integration/test_api.py",
        ]
    },
}

print("Dependency rules defined for", len(DEPENDENCY_RULES), "prefixes")

## Cell 13 — Change 1: Add due date and overdue listing

In [ ]:
CHANGE_1 = RequirementChange(
    change_id="CH-001",
    version="1.1",
    title="Add due date and overdue listing",
    requirement_delta="""
Tasks may have an optional due date in ISO format YYYY-MM-DD.
Add an endpoint GET /tasks/overdue?current_date=YYYY-MM-DD.
A task is overdue when:
1. it has a due date,
2. it is not completed,
3. due_date is strictly earlier than current_date.
""".strip(),
    dsl_patch={
        "project.version": "1.1",
        "entities.Task.fields.due_date": {
            "type": "date",
            "required": False,
            "nullable": True,
        },
        "actions.create_task.inputs": ["title", "due_date"],
        "actions.list_overdue_tasks": {
            "inputs": ["current_date"],
            "output": "List[Task]",
        },
        "business_rules.overdue_rule": {
            "text": (
                "A task is overdue when it has a due date, "
                "is incomplete, and due_date is earlier "
                "than current_date."
            )
        },
    },
    expected_change_types=[
        "add_field",
        "change_action_signature",
        "add_endpoint",
        "add_business_rule",
    ],
    allowed_files=[
        "app/models.py",
        "app/schemas.py",
        "app/service.py",
        "app/main.py",
        "tests/unit/test_service.py",
        "tests/integration/test_api.py",
        "dsl.yaml",
    ],
    hidden_test_files={
        "hidden_tests/test_change_001.py": '''
from fastapi.testclient import TestClient

from app.main import app


client = TestClient(app)


def test_completed_task_is_not_overdue() -> None:
    created = client.post(
        "/tasks",
        json={
            "title": "Completed task",
            "due_date": "2026-01-01",
        },
    ).json()

    client.post(f"/tasks/{created['id']}/complete")

    response = client.get(
        "/tasks/overdue",
        params={"current_date": "2026-06-01"},
    )

    returned_ids = {
        item["id"]
        for item in response.json()
    }

    assert created["id"] not in returned_ids


def test_task_due_today_is_not_yet_overdue() -> None:
    created = client.post(
        "/tasks",
        json={
            "title": "Due today",
            "due_date": "2026-06-01",
        },
    ).json()

    response = client.get(
        "/tasks/overdue",
        params={"current_date": "2026-06-01"},
    )

    returned_ids = {
        item["id"]
        for item in response.json()
    }

    assert created["id"] not in returned_ids
''',
    },
)

print(CHANGE_1.change_id, CHANGE_1.title)

## Cell 14 — Run Change 1

In [ ]:
import shutil

RESULTS_PATH = PATHS["results"] / "selective_runs.jsonl"

change_1_result = run_selective_change(
    project_dir=PROJECT_DIR,
    change=CHANGE_1,
    dependency_rules=DEPENDENCY_RULES,
    tokenizer=tokenizer,
    model=model,
    config=CONFIG,
    results_path=RESULTS_PATH,
    snapshots_dir=PATHS["snapshots"],
)

print(json.dumps(
    {
        "change_id": change_1_result["change_id"],
        "version": change_1_result["version"],
        "passed": change_1_result["passed"],
        "first_pass_acceptance": change_1_result["first_pass_acceptance"],
        "local_repair_used": change_1_result["local_repair_used"],
        "accepted_after_local_repair": change_1_result["accepted_after_local_repair"],
        "llm_repair_used": change_1_result["llm_repair_used"],
        "total_tokens": change_1_result["metrics"]["total_tokens"],
    },
    indent=2,
))

## Cell 15 — Verify Change 1 commit

In [ ]:
if change_1_result["passed"]:
    committed_dsl = read_yaml(PROJECT_DIR / "dsl.yaml")
    print("Committed DSL version:", committed_dsl["project"]["version"])

    committed_validation = validate_project(
        PROJECT_DIR,
        test_timeout=CONFIG.test_timeout_seconds,
    )
    assert committed_validation["all_passed"], "Committed project fails validation"
    print("Change 1 committed successfully.")
else:
    print("Change 1 FAILED and was NOT committed.")

## Cell 16 — Change 2: Add task tags and filter by tag

In [ ]:
CHANGE_2 = RequirementChange(
    change_id="CH-002",
    version="1.2",
    title="Add task tags and filter tasks by tag",
    requirement_delta="""
Tasks may have zero or more tags.

Requirements:
1. Add an optional tags field to Task.
2. The tags field is a list of strings.
3. If tags are not provided when creating a task, the task must use
   an empty list.
4. Each Task instance must have its own independent tags list.
5. POST /tasks must accept tags.
6. Task responses must include tags.
7. GET /tasks must support an optional query parameter named tag.
8. When tag is provided, return only tasks containing that exact tag.
9. Tag matching is case-sensitive.
10. When tag is omitted, GET /tasks must preserve its existing
    behavior and return all tasks.
11. Existing due-date, overdue, completion, and title behavior must
    remain unchanged.
""".strip(),
    dsl_patch={
        "project.version": "1.2",
        "entities.Task.fields.tags": {
            "type": "list[string]",
            "required": False,
            "default": [],
        },
        "actions.create_task.inputs": ["title", "due_date", "tags"],
        "actions.list_tasks.inputs": ["tag"],
        "business_rules.tag_filter_rule": {
            "text": (
                "When the optional tag query parameter is provided, "
                "list_tasks returns only tasks containing that exact "
                "case-sensitive tag. When tag is omitted, all tasks "
                "are returned."
            )
        },
    },
    expected_change_types=[
        "add_field",
        "change_action_signature",
        "add_optional_query_filter",
        "add_business_rule",
    ],
    allowed_files=[
        "app/models.py",
        "app/schemas.py",
        "app/service.py",
        "app/main.py",
        "tests/unit/test_service.py",
        "tests/integration/test_api.py",
        "dsl.yaml",
    ],
    hidden_test_files={
        "hidden_tests/test_change_002.py": '''
from fastapi.testclient import TestClient

from app.main import app


client = TestClient(app)


def create_task(
    title: str,
    tags: list[str] | None = None,
) -> dict:
    payload: dict = {"title": title}
    if tags is not None:
        payload["tags"] = tags
    response = client.post("/tasks", json=payload)
    assert response.status_code == 201
    return response.json()


def test_task_without_tags_has_empty_list() -> None:
    task = create_task("No tags")
    assert task["tags"] == []


def test_create_task_preserves_tags() -> None:
    task = create_task("Tagged task", ["work", "urgent"])
    assert task["tags"] == ["work", "urgent"]


def test_filter_tasks_by_existing_tag() -> None:
    work_task = create_task("Work task", ["work", "urgent"])
    create_task("Personal task", ["personal"])

    response = client.get("/tasks", params={"tag": "work"})
    assert response.status_code == 200
    returned_ids = {item["id"] for item in response.json()}
    assert work_task["id"] in returned_ids
    assert len(response.json()) == 1


def test_filter_is_case_sensitive() -> None:
    create_task("Uppercase tag", ["Work"])
    response = client.get("/tasks", params={"tag": "work"})
    assert response.status_code == 200
    assert response.json() == []


def test_list_without_tag_returns_all_tasks() -> None:
    create_task("First task", ["work"])
    create_task("Second task", ["personal"])
    response = client.get("/tasks")
    assert response.status_code == 200
    assert len(response.json()) == 2


def test_existing_overdue_behavior_still_works() -> None:
    created = client.post(
        "/tasks",
        json={"title": "Old tagged task", "due_date": "2026-01-01", "tags": ["work"]},
    )
    assert created.status_code == 201
    response = client.get("/tasks/overdue", params={"current_date": "2026-06-01"})
    assert response.status_code == 200
    returned_ids = {item["id"] for item in response.json()}
    assert created.json()["id"] in returned_ids
''',
    },
)

print(CHANGE_2.change_id, CHANGE_2.title)

## Cell 17 — Run Change 2

In [ ]:
change_2_result = run_selective_change(
    project_dir=PROJECT_DIR,
    change=CHANGE_2,
    dependency_rules=DEPENDENCY_RULES,
    tokenizer=tokenizer,
    model=model,
    config=CONFIG,
    results_path=RESULTS_PATH,
    snapshots_dir=PATHS["snapshots"],
)

print(json.dumps(
    {
        "change_id": change_2_result["change_id"],
        "version": change_2_result["version"],
        "passed": change_2_result["passed"],
        "first_pass_acceptance": change_2_result["first_pass_acceptance"],
        "local_repair_used": change_2_result["local_repair_used"],
        "accepted_after_local_repair": change_2_result["accepted_after_local_repair"],
        "llm_repair_used": change_2_result["llm_repair_used"],
        "total_tokens": change_2_result["metrics"]["total_tokens"],
    },
    indent=2,
))

## Cell 18 — Verify Change 2 commit

In [ ]:
if change_2_result["passed"]:
    committed_dsl = read_yaml(PROJECT_DIR / "dsl.yaml")
    print("Committed DSL version:", committed_dsl["project"]["version"])

    committed_validation = validate_project(
        PROJECT_DIR,
        test_timeout=CONFIG.test_timeout_seconds,
    )
    assert committed_validation["all_passed"], "Committed project fails validation"
    print("Change 2 committed successfully.")
else:
    print("Change 2 FAILED and was NOT committed.")

## Cell 19 — Change 3: Overdue includes current date

In [ ]:
CHANGE_3 = RequirementChange(
    change_id="CH-003",
    version="1.3",
    title="Overdue includes current date",
    requirement_delta="""
Change the overdue rule so that tasks due on the current
date are also considered overdue.

Previously a task was overdue only when due_date was strictly
earlier than current_date. Now a task is overdue when
due_date is earlier than or equal to current_date.

All other behavior (completed tasks, tags, filtering) must
remain unchanged.
""".strip(),
    dsl_patch={
        "project.version": "1.3",
        "business_rules.overdue_rule": {
            "text": (
                "A task is overdue when it has a due date, "
                "is incomplete, and due_date is earlier than "
                "or equal to current_date."
            )
        },
    },
    expected_change_types=[
        "modify_business_rule",
    ],
    allowed_files=[
        "app/service.py",
        "app/main.py",
        "app/models.py",
        "app/schemas.py",
        "tests/unit/test_service.py",
        "tests/integration/test_api.py",
        "dsl.yaml",
    ],
    hidden_test_files={
        "hidden_tests/test_change_001.py": '''
from fastapi.testclient import TestClient

from app.main import app


client = TestClient(app)


def test_completed_task_is_not_overdue() -> None:
    created = client.post(
        "/tasks",
        json={
            "title": "Completed task",
            "due_date": "2026-01-01",
        },
    ).json()

    client.post(f"/tasks/{created['id']}/complete")

    response = client.get(
        "/tasks/overdue",
        params={"current_date": "2026-06-01"},
    )

    returned_ids = {
        item["id"]
        for item in response.json()
    }

    assert created["id"] not in returned_ids
''',
        "hidden_tests/test_change_003.py": '''
from fastapi.testclient import TestClient

from app.main import app


client = TestClient(app)


def test_task_due_today_is_now_overdue() -> None:
    created = client.post(
        "/tasks",
        json={
            "title": "Due today",
            "due_date": "2026-06-01",
        },
    ).json()

    response = client.get(
        "/tasks/overdue",
        params={"current_date": "2026-06-01"},
    )

    returned_ids = {
        item["id"]
        for item in response.json()
    }

    assert created["id"] in returned_ids


def test_task_due_yesterday_is_overdue() -> None:
    created = client.post(
        "/tasks",
        json={
            "title": "Due yesterday",
            "due_date": "2026-05-31",
        },
    ).json()

    response = client.get(
        "/tasks/overdue",
        params={"current_date": "2026-06-01"},
    )

    returned_ids = {
        item["id"]
        for item in response.json()
    }

    assert created["id"] in returned_ids


def test_task_due_tomorrow_is_not_overdue() -> None:
    created = client.post(
        "/tasks",
        json={
            "title": "Due tomorrow",
            "due_date": "2026-06-02",
        },
    ).json()

    response = client.get(
        "/tasks/overdue",
        params={"current_date": "2026-06-01"},
    )

    returned_ids = {
        item["id"]
        for item in response.json()
    }

    assert created["id"] not in returned_ids


def test_completed_task_due_today_not_overdue() -> None:
    created = client.post(
        "/tasks",
        json={
            "title": "Completed due today",
            "due_date": "2026-06-01",
        },
    ).json()

    client.post(f"/tasks/{created['id']}/complete")

    response = client.get(
        "/tasks/overdue",
        params={"current_date": "2026-06-01"},
    )

    returned_ids = {
        item["id"]
        for item in response.json()
    }

    assert created["id"] not in returned_ids
''',
    },
)

print(CHANGE_3.change_id, CHANGE_3.title)

## Cell 20 — Run Change 3

In [ ]:
change_3_result = run_selective_change(
    project_dir=PROJECT_DIR,
    change=CHANGE_3,
    dependency_rules=DEPENDENCY_RULES,
    tokenizer=tokenizer,
    model=model,
    config=CONFIG,
    results_path=RESULTS_PATH,
    snapshots_dir=PATHS["snapshots"],
)

print(json.dumps(
    {
        "change_id": change_3_result["change_id"],
        "version": change_3_result["version"],
        "passed": change_3_result["passed"],
        "first_pass_acceptance": change_3_result["first_pass_acceptance"],
        "local_repair_used": change_3_result["local_repair_used"],
        "accepted_after_local_repair": change_3_result["accepted_after_local_repair"],
        "llm_repair_used": change_3_result["llm_repair_used"],
        "total_tokens": change_3_result["metrics"]["total_tokens"],
    },
    indent=2,
))

## Cell 21 — Verify Change 3 commit

In [ ]:
if change_3_result["passed"]:
    committed_dsl = read_yaml(PROJECT_DIR / "dsl.yaml")
    print("Committed DSL version:", committed_dsl["project"]["version"])

    committed_validation = validate_project(
        PROJECT_DIR,
        test_timeout=CONFIG.test_timeout_seconds,
    )
    assert committed_validation["all_passed"], "Committed project fails validation"
    print("Change 3 committed successfully.")
else:
    print("Change 3 FAILED and was NOT committed.")

## Cell 22 — Summary of all changes

In [ ]:
print("=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)

all_results = [
    change_1_result,
    change_2_result,
    change_3_result,
]

for result in all_results:
    m = result["metrics"]
    print(
        f"\n{result['change_id']:8s} | "
        f"v{result['version']:4s} | "
        f"{'PASS' if result['passed'] else 'FAIL':4s} | "
        f"first_pass={result['first_pass_acceptance']} | "
        f"local_repair={result['local_repair_used']} | "
        f"llm_repair={result['llm_repair_used']} | "
        f"tokens={m['total_tokens']}"
    )

total_tokens = sum(r["metrics"]["total_tokens"] for r in all_results)
passed_count = sum(1 for r in all_results if r["passed"])

print(f"\n{'=' * 70}")
print(f"Total tokens: {total_tokens}")
print(f"Passed: {passed_count}/{len(all_results)}")
print(f"{'=' * 70}")

## Cell 23 — Show recorded runs from JSONL

In [ ]:
recorded_runs = load_metrics_jsonl(RESULTS_PATH)

print(f"Recorded runs: {len(recorded_runs)}")

for run in recorded_runs:
    print(
        run.get("change_id"),
        "| passed:", run.get("final_acceptance"),
        "| tokens:", run.get("total_tokens"),
        "| category:", run.get("failure_category"),
    )

## Cell 24 — Zip output

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d-%H%M")
filename = f"selective_regeneration-{timestamp}"

archive_name = shutil.make_archive(
    filename,
    "zip",
    root_dir=".",
    base_dir="selective_regeneration",
)

print(f"Archive created at: {archive_name}")